# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join, exists
from tqdm.auto import tqdm

# Import Functions
sys.path.append("../../")

from src.configs.blood_config import data_name, data_name_ood_in, data_name_ood_out, data_name_ood_diff
from src.file_manager.filepath import FilePath
from src.file_manager.load_save_df import load_all_pred_dfs
from src.data_generator.blood import load_bloodmnist_data_dict
from src.data_generator.raabin import load_raabin_data_dict
from src.data_generator.bonemarrow import load_bonemarrow_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets
from src.evaluation.generate_expl import get_incorrect_test_predictions, get_explanation_heatmap, show_explanation, get_only_explanations

from model_ue_dict import ModelClass_dict, ue_dict
from cur_seed import seed
seed = 2024

ue_col = "egrue"
display_ue_col = "percentile_ue"

fp = FilePath(data_name=data_name, seed=seed)
fp_ood_in = FilePath(data_name=data_name_ood_in, seed=seed)
fp_ood_out = FilePath(data_name=data_name_ood_out, seed=seed)
fp_ood_diff = FilePath(data_name=data_name_ood_diff, seed=seed)

fp = FilePath(data_name=data_name, seed=seed)

# Load Data

In [ ]:
data_dict = load_bloodmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())

# Load Predictions

In [ ]:
pred_df = load_all_pred_dfs(fp, ModelClass_dict=ModelClass_dict)
def min_max_norm_col(df, col):
    df = df.copy()
    df[col] = (df[col]-df[col].min())/(df[col].max()-df[col].min())
    return df
pred_df[display_ue_col] = pred_df[ue_col].rank(pct=True) * 100
pred_df = pred_df[pred_df["split_perf"]!="Train"]

# Get Examples
 - Use Valid/Test Dataset
 - Separate Correct and Incorrect Istances
 - Within Each Group Separate High Uncertainty vs Low Uncertainty (Confidence)

In [ ]:
def get_user_study_examples(
    pred_df, split_col="split_perf", included_splits=["Valid", "Test-Blood"],
    correct_col="correct_resnet", ue_col=display_ue_col, num_instance_per_cat=100
):
    df_dict = {}

    # 1. Only Valid/Test Datset
    eval_df = pred_df[pred_df[split_col].isin(included_splits)]

    # 2. Separate Correct and Incorrect
    correctness_dict = {"correct": eval_df[eval_df[correct_col]],
        "incorrect": eval_df[~eval_df[correct_col]]}

    # 3. Within Each Group Separate High Uncertainty vs Low Uncertainty (Confidence)
    correctness_labels = ("correct", "incorrect")
    uncertainty_labels = ("high_uncertainty", "low_uncertainty")
    for correctness in correctness_labels:
        correct_df = correctness_dict[correctness]
        for uncertainty in uncertainty_labels:
            ascending = uncertainty == "low_uncertainty"
            cur_correct_df = correct_df.copy().sort_values(by=ue_col, ascending=ascending)
            df_dict[correctness+"-"+uncertainty] = cur_correct_df.head(num_instance_per_cat).reset_index()
    return df_dict

user_study_df_dict = get_user_study_examples(pred_df)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from os import makedirs, listdir
import textwrap
from math import ceil
import numpy as np

def get_uncertainty_label(display_ue):
    uncertainty_label = "Low"
    if display_ue > 100*2/3:
        uncertainty_label = "High"
    elif display_ue > 100/3:
        uncertainty_label = "Medium"
    return uncertainty_label

def plot_image(image, overlay=None, ax=None, cmap="hot", alpha=1):
    # Plot Image + UX
    if ax is None:
        fig, ax = plt.figure(figsize=(6, 6), dpi=300)
    ax.imshow(image)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if overlay is not None:
        im = ax.imshow(overlay, cmap=cmap, vmin=0, vmax=1, alpha=alpha)
        return im

def output_examples(
    user_study_df_dict, selected_instances_dict, data_dict, description_dict,
    split_col="split", index_col="index", target_col='class', pred_col="class_pred_label_resnet",
    display_ue_col=display_ue_col, rounding=3, seed=seed, alpha=1, c=1, gamma=1, dpi=300, size=2, cmap="hot",
    fp_saved_folder = f"user_study_selected"
):
    convert_split = {"Valid": "val_df", "Test": "test_df"}
    classes = data_dict['classes']
    cat_key, idx_key = "cat", "idx"
    expl_key, image_key = "egRUE", "Image"
    actual_label_key, pred_label_key = "actual_label", "pred_label"
    ue_key, ue_label_key, interpretation_key = "ue", "ue_label", "interpretation"
    # Get Explanations
    print("Computing Explanations...")
    all_explanations = []
    for cat in tqdm(user_study_df_dict.keys(), total=len(user_study_df_dict)):
        cur_df = user_study_df_dict[cat]
        selected_idx = selected_instances_dict[cat]
        cur_desc_list = description_dict[cat]
        for idx, interpretation in tqdm(zip(selected_idx, cur_desc_list), total=len(selected_idx)):
            # Get Info
            row = cur_df.iloc[idx]
            split = convert_split[row[split_col]]
            index = row[index_col]
            expl_dict = get_only_explanations(data_dict, split, index, fp, seed, gamma=gamma)
            expl_dict[actual_label_key] = classes[row[target_col]].capitalize()
            expl_dict[pred_label_key] = classes[row[pred_col]].capitalize()
            expl_dict[ue_key] = round(row[display_ue_col], rounding)
            expl_dict[ue_label_key] = get_uncertainty_label(expl_dict[ue_key])
            expl_dict[interpretation_key] = textwrap.fill(interpretation, width=45)
            expl_dict[cat_key] = cat
            expl_dict[idx_key] = str(idx)
            all_explanations.append(expl_dict)

    # Normalise Explanations
    print("Normalising...")
    expl_min = np.min([expl_dict[expl_key] for expl_dict in all_explanations])
    expl_max = np.max([expl_dict[expl_key] for expl_dict in all_explanations])
    for i in tqdm(range(len(all_explanations))):
        expl = all_explanations[i][expl_key]
        all_explanations[i][expl_key] = (expl-expl_min)/(expl_max-expl_min)

    # Plot Explanations
    print("Plotting Explanations...")
    for i in tqdm(range(len(all_explanations))):
        expl_dict = all_explanations[i]
        img = expl_dict[image_key]
        expl = expl_dict[expl_key]
        cat = expl_dict[cat_key]
        idx = expl_dict[idx_key]
        actual_label = expl_dict[actual_label_key]
        pred_label = expl_dict[pred_label_key]
        ue = expl_dict[ue_key]
        ue_label = expl_dict[ue_label_key]
        interpretation = expl_dict[interpretation_key]
        fp_cur_idx = join(fp_saved_folder, cat, idx)
        if not exists(fp_cur_idx):
            makedirs(fp_cur_idx)
        # 1. Plot Only Image
        fig1, ax1 = plt.subplots(1, 1, figsize=(size, size), dpi=dpi)
        plot_image(img, ax=ax1)
        # - Add Pred and UE into Figure
        ax1.text(
            0, -0.25, f"Prediction: {pred_label}\nUncertainty: {ue}% ({ue_label})", 
            transform=ax1.transAxes, ha="left", color='black')
        fig1.savefig(join(fp_cur_idx, f"{cat}_{idx}_pred_ue_only.png"), bbox_inches="tight")
        plt.close()

        # 2. Plot Only Image + UX
        n_cols = 2
        fig2, axes = plt.subplots(1, n_cols, figsize=(size*n_cols, size), dpi=dpi)
        plot_image(img, ax=axes[0])
        im = plot_image(img, overlay=expl, ax=axes[1], alpha=alpha, cmap=cmap)
        # - Add colorbar
        cbar = fig2.colorbar(im, ax=axes.ravel().tolist(),
            ticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],shrink=0.7)
        cbar.ax.tick_params(labelsize=5, pad=1, length=2)
        cbar.set_label('Contribution to Uncertainty', 
            rotation=270, labelpad=10, fontsize=5)
        # - Add Pred and UE into Figure
        axes[0].text(
            0, -0.3, f"Prediction: {pred_label}\nUncertainty: {ue}% ({ue_label})", 
            transform=axes[0].transAxes, ha="left", color='black')
        # Add Interpretation
        axes[0].text(0, -0.5, 
            "Uncertainty Explanation Interpretation:",
            transform=axes[0].transAxes, ha="left", fontweight='bold', color='black')
        axes[0].text(
            0, -0.65-0.15*(interpretation.count("\n")),  
            interpretation, transform=axes[0].transAxes, ha="left", color='black')
        fig2.savefig(join(fp_cur_idx, f"{cat}_{idx}_pred_ue_ux.jpg"), bbox_inches="tight")
        plt.close()
        # Save Information
        text_desc = (
            f"Index: {idx}, Split: {split}, Index: {index}, Actual: {actual_label}\n"
            f"Prediction: {pred_label}, Uncertainty: {ue:.3f}% ({ue_label})\n"
        )
        with open(join(fp_cur_idx, "info.txt"), "w") as f:
            f.write(text_desc)

def display_all_examples(fp_folder, n_cols=3, size=1, dpi=300):
    images = []
    for example_type in listdir(fp_folder):
        fp_example_type = join(fp_folder, example_type)
        for index in listdir(fp_example_type):
            fn_image = f"{example_type}_{index}_pred_ue_ux.jpg"
            fp_image = join(fp_example_type, index, fn_image)
            img = mpimg.imread(fp_image)
            images.append(img)
    n_images = len(images)
    n_rows = ceil(n_images/n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*size, n_rows*size*0.7), dpi=dpi)
    axes = axes.flatten()
    for i, image in enumerate(images):
        axes[i].imshow(image)
        axes[i].axis('off')
    plt.subplots_adjust(wspace=0, hspace=0)
    plt.tight_layout()
    plt.show()

all_instances_dict = {  # list(range(100))
    'correct-high_uncertainty': [],
    'incorrect-high_uncertainty': [2, 3, 5],
    'correct-low_uncertainty': [4, 7, 11],
    'incorrect-low_uncertainty': [4, 7, 17],
}
description_dict = {
    'correct-high_uncertainty': [],
    'incorrect-high_uncertainty': [
        "The model is unsure about the granularity at the cell boundary and nucleus boundary.", 
        "The model is confused by the presence of surrounding cells.", 
        "The model is unsure about the nuclear boundary."],
    'correct-low_uncertainty': [
        "The model is unsure about the cell boundary and nuclear boundary.", 
        "The model is unsure about the cell boundary and nuclear boundary.", 
        "The model is unsure about the cell boundary and nuclear boundary."
    ],
    'incorrect-low_uncertainty': [
        "The model is unsure about most part of the cell, especially its cytoplasm.", 
        "The model is unsure about structure of the nucleus.", 
        "The model is unsure about the structure of the nucleus."
    ],
}
import matplotlib.colors as mcolors
cmap_transparent_red = mcolors.LinearSegmentedColormap.from_list(
    "TransparentRed", 
    [(1, 0, 0, 0), (1, 0, 0, 1)]  # (R, G, B, Alpha) -> From 0% opacity to 100% opacity
)
cmap_transparent_neon_red = mcolors.LinearSegmentedColormap.from_list(
    "TransparentRed", 
    [(1, 0, 0, 0), (1.0, 0.6, 0.6)]  # (R, G, B, Alpha) -> From 0% opacity to 100% opacity
)
cmap_black_red = mcolors.LinearSegmentedColormap.from_list("BlackRed", ["black", "red"])
cmap_neon_red = mcolors.LinearSegmentedColormap.from_list(
    "NeonRed", ["black", (0.8, 0, 0), (1.0, 0.6, 0.6)]
)
cmap_black_red = mcolors.LinearSegmentedColormap.from_list(
    "BlackRed", [(0, 0, 0, 1), (1, 0, 0, 1)]
)
output_examples(
    user_study_df_dict, 
    all_instances_dict, 
    data_dict, 
    description_dict=description_dict, 
    alpha=0.8, gamma=0.6, cmap=cmap_neon_red)
display_all_examples(fp_folder="user_study_selected")

In [ ]:
all_instances_dict.keys()